# Solvers used in this book

To get started in a Jupyter or Colab notebook, run this magic command in a code cell:

```ipython
%pip install pyomo highspy
```

`%pip` installs into the current notebook kernel's environment. If you are installing from a terminal instead, use:

```console
python -m pip install pyomo highspy
```

This provides the solver setup for most examples in chapters 1–4, 7, 9 and 10, and the linear parts of chapters 5 and 8. Specific examples need additional solvers, as shown below; their preambles also identify other Python packages they use. The choice follows the mathematical formulation: changing a solver name cannot make an incompatible formulation solvable.

Use **Run all** to execute the setup and integer-solution check below. This notebook needs no commercial solver license.

### Modeling tools, solvers and interfaces

**Pyomo** describes the variables, objective and constraints; a **solver** computes a solution. For setup instructions, see the [Pyomo installation guide](https://www.pyomo.org/installation). The package `highspy` supplies HiGHS's Python interface.

**CVXPY** is another modeling tool, used in the refinery example. It also calls a solver underneath. Its [solver guide](https://www.cvxpy.org/tutorial/solvers/index.html) shows how to select that solver and inspect the results.

Names such as `appsi_highs`, `mosek_direct` and `gurobi_persistent` identify **Pyomo interfaces**: the connections through which Pyomo sends a model to a solver and retrieves its results. An interface determines which solver features are accessible from that code.


## Solver choices and notebook map

Choosing a solver starts with understanding the model. Are its expressions linear? Do some decisions have to be integer? Is the continuous problem convex, and can its structure be expressed with cones? Sometimes a suitable formulation removes the need for integer optimization: with a totally unimodular constraint matrix, integral right-hand sides and integral bounds, a linear relaxation can have an integral optimal vertex. [Fleet assignment (§7.1)](../07/01-fleet-assignment.ipynb) discusses this possibility. In other models, such as installation decisions in [building insulation (§6.4)](../06/04-building-insulation.ipynb), the solver needs to enforce the binary choices. Formulation and solver selection therefore inform each other: expressing useful structure can make more algorithms available while preserving the intended decisions.

**LO** means linear optimization; **MILO** means mixed-integer linear optimization. The classes below describe how the book uses each solver, rather than listing every feature the software offers. A selected solver and its Python interface are separate choices: callbacks, solver options and explicit cone components can require a particular interface.

### Solvers selected in the notebooks

| Solver | Role in these examples | Where to see it |
|---|---|---|
| [HiGHS](https://highs.dev/) | LO and MILO; also the linear master and subproblems in several decomposition methods | [Production planning (§1.2)](../01/02-production-planning-basic.ipynb), [shift scheduling (§3.2)](../03/02-shift-scheduling.ipynb), [minimum-cost flow (§4.2)](../04/02-mincost-flow.ipynb), [economic dispatch (§9.4)](../09/04-economic-dispatch.ipynb), and both the Pyomo and CVXPY models in the [refinery extra](../05/05-refinery-production.ipynb). The usual Pyomo interface is `appsi_highs`. |
| [Ipopt](https://coin-or.github.io/Ipopt/) | Smooth continuous nonlinear optimization; a local method for nonconvex models | [Milk pooling (§5.1)](../05/01-milk-pooling.ipynb) and the discretized dynamic model in [Functional programming with Pyomo](functional-programming-pyomo.ipynb). It is also the Colab selection for the continuous quadratic and conic examples listed below. The interface is `ipopt`. |
| [SCIP](https://www.scipopt.org/doc/html/WHATPROBLEMS.php) | Nonlinear optimization with or without integer decisions, including global bounds for nonconvex subproblems | [Building insulation (§6.4)](../06/04-building-insulation.ipynb), [robust BIM (§8.1)](../08/01-bim-robust-optimization.ipynb) and the nonlinear pattern-improvement models in [cutting stock](../05/06-cutting-stock.ipynb) use `scip_direct` through PySCIPOpt. [Two-stage production (§10.2)](../10/02-two-stage-production-planning.ipynb) uses PySCIPOpt directly for its continuous, nonconvex worst-case subproblem. HiGHS handles the linear parts of these examples where needed. |
| [MOSEK](https://docs.mosek.com/latest/intro/overview.html) | Convex quadratic and conic optimization; mixed-integer conic optimization in optional comparisons | The local selection for the continuous examples below. [§6.4](../06/04-building-insulation.ipynb) and [§8.1](../08/01-bim-robust-optimization.ipynb) retain optional MOSEK conic comparisons, switched off by default. The usual interface is `mosek_direct`; §5.4 uses `mosek`. |
| [CBC](https://coin-or.github.io/Cbc/intro.html) | LO and MILO through the `cbc` executable interface | A production model in [Functional programming with Pyomo](functional-programming-pyomo.ipynb), the CBC demonstration and Colab benchmarks in [facility location (§3.6)](../03/06-facility-location.ipynb), and the solver comparison in [dinner seating (§4.1)](../04/01-dinner-seat-allocation.ipynb). |
| [Gurobi](https://docs.gurobi.com/projects/optimizer/en/current/concepts/modeling/constraints.html) | LO/MILO comparisons and persistent-model/callback examples | [Facility location (§3.6)](../03/06-facility-location.ipynb) and [dinner seating (§4.1)](../04/01-dinner-seat-allocation.ipynb) use `gurobi_direct`; the [traveling-salesman extra](../04/08-traveling-salesman-problem.ipynb) uses `gurobi_persistent` alongside HiGHS. Gurobi is required for those persistent-interface sections. |
| [CPLEX](https://www.ibm.com/docs/en/icos/22.1.0?topic=cplex-types-problems-solved) | LO/MILO solver comparisons | [Facility location (§3.6)](../03/06-facility-location.ipynb), through `cplex_direct`. Its local benchmark selects CPLEX and Gurobi. |
| [Xpress](https://github.com/fico-xpress/xpress-training/blob/main/python/md/python-full-course.md) | LO/MILO solver comparisons | The comparison over available solvers in [facility location (§3.6)](../03/06-facility-location.ipynb), through `xpress_direct`. |

HiGHS, Ipopt, SCIP and CBC are open source; MOSEK, Gurobi, CPLEX and Xpress are commercial products. See the [license and academic-access guidance](installing-pyomo-and-solvers.ipynb#licenses-and-academic-access) for their terms and the available no-cost routes.

#### Continuous quadratic and conic examples

These notebooks select **MOSEK when run locally** and **Ipopt on Colab**. MOSEK can use explicit cone structure; Ipopt receives continuous nonlinear expressions. This distinction concerns the formulation and algorithm, even when both solve the same convex problem.

| Notebooks | Structure and setup |
|---|---|
| [OLS regression (§5.2)](../05/02-ols-regression.ipynb), [Markowitz portfolio (§5.3)](../05/03-markowitz-portfolio.ipynb), [binary SVM (§5.4)](../05/04-svm-binary-classification.ipynb) | Quadratic models use MOSEK/Ipopt. §§5.2 and 5.4 also use HiGHS for their linear formulations. |
| [Economic order quantity (§6.1)](../06/01-economic-order-quantity.ipynb) | Continuous conic reformulations of reciprocal order and inventory costs. |
| [Kelly criterion (§6.2)](../06/02-kelly-criterion.ipynb), [investment wheel](../06/06-investment-wheel.ipynb), [optimal growth portfolios](../06/07-optimal-growth-portfolios.ipynb) | Continuous models using exponential cones or their nonlinear expressions. |
| [Markowitz revisited (§6.3)](../06/03-markowitz-portfolio-revisited.ipynb), [chance-constrained portfolio (§9.1)](../09/01-markowitz-portfolio-with-chance-constraint.ipynb) | Quadratic/conic formulations of portfolio risk. |
| [Conic SVM extra](../06/05-svm-conic.ipynb) | The Ipopt path explicitly selects MA57 for the dense kernel model. MA57 is an internal linear-equation solver, not a separate solver for the optimization model; see [Ipopt's linear-solver options](https://coin-or.github.io/Ipopt/OPTIONS.html#OPT_linear_solver). |

The binary layer choices in §6.4 follow the SCIP route described above, with the optional MOSEK comparison at the end of that notebook. They are not part of the continuous Ipopt selection in this table.

**CVXPY is a modeling tool.** In the refinery example, `problem.solve(solver=cp.HIGHS)` selects HiGHS explicitly. Installing CVXPY adds another way to express the model; it does not introduce a different optimization algorithm into that comparison.

### Other solvers mentioned in this guide

The following solvers are not selected by the current example code. They are included to explain alternatives a reader may encounter, including tools supplied with solver distributions. The applications in the final column are suggestions based on the mathematical problem class, not additional implementations or tested replacements in these notebooks.

| Solver | Applicable problem classes | A relevant application in the book |
|---|---|---|
| [GLPK](https://www.gnu.org/software/glpk/) | LO and MILO | An alternative for linear production planning such as [§1.2](../01/02-production-planning-basic.ipynb), network-flow models such as [§4.2](../04/02-mincost-flow.ipynb), or linear integer scheduling such as [§3.2](../03/02-shift-scheduling.ipynb). Pyomo's `glpk` interface requires the separate `glpsol` executable. |
| [Bonmin](https://coin-or.github.io/Bonmin/Intro.html) | Smooth convex mixed-integer nonlinear optimization; its nonconvex use is heuristic | A possible comparison for [multilayer insulation (§6.4)](../06/04-building-insulation.ipynb), retaining binary installation decisions and expressing the thermal constraint in convex form, for example $U \geq 1/R$ with $R>0$. Simply passing the existing bilinear expression to Bonmin would not establish the convexity assumptions used by its algorithms. |
| [Couenne](https://www.coin-or.org/Couenne/) | Global optimization of supported nonconvex nonlinear models, including mixed-integer models | The bilinear pattern-improvement problems in [cutting stock](../05/06-cutting-stock.ipynb), or a global-solution comparison for the nonconvex pooling model in [§5.1](../05/01-milk-pooling.ipynb). Cutting stock selects SCIP; pooling uses Ipopt to illustrate local nonlinear optimization. |

Trying an alternative means matching the model representation, interface and options, then checking feasibility and termination. For a global method, inspect the incumbent and bound or gap as well. A solver's mathematical scope does not establish that a particular binary distribution, notebook setup or callback implementation will work unchanged.


### Why can a nonlinear solver solve a convex model?

“Nonlinear” describes the expressions; “convex” describes the geometry. A model can be both. For example, minimizing a positive-semidefinite quadratic objective over linear constraints is convex nonlinear optimization. As the [Ipopt documentation](https://coin-or.github.io/Ipopt/) explains, Ipopt works with smooth continuous expressions. For a convex problem, a local minimum is also global.

A conic solver such as MOSEK instead receives the cone structure explicitly. Expressing a model in conic form lets its algorithms use that structure. This is why chapter 6 revisits problems that can also be written as nonlinear models.

Integer decisions need a solver that enforces them. A binary variable represents a yes/no choice; allowing a value such as 0.3 produces a continuous relaxation with different feasible decisions. Ipopt solves continuous problems, so integer choices require a mixed-integer solver.

## Licenses and academic access

“Open source”, “free of charge” and “academic use” describe different things. Open-source licenses grant rights to inspect, modify and redistribute software under stated conditions. A free community edition or academic license of a commercial solver grants the rights in the vendor's agreement; it does not make that solver open source.

### Open-source terms

[HiGHS uses MIT](https://highs.dev/) and [SCIP uses Apache 2.0](https://github.com/scipopt/scip/blob/master/LICENSE), both permissive licenses. [Ipopt](https://github.com/coin-or/Ipopt/blob/stable/3.14/LICENSE), [CBC](https://github.com/coin-or/Cbc/blob/master/LICENSE), [Couenne](https://github.com/coin-or/Couenne/blob/master/README.md) and [Bonmin](https://github.com/coin-or/Bonmin) use the Eclipse Public License (EPL). These licenses allow commercial use, subject to their conditions. Keep the applicable notices and license texts when redistributing; copyleft licenses can also require providing source for covered software or modifications. Check the license shipped with your installed release and its dependencies: an open-source solver can include separately licensed numerical libraries, such as HSL in some Ipopt builds.

**GLPK uses GNU GPL, a copyleft license. Using it does not automatically require publishing your model, data or all your application code.** GLPK's author explicitly distinguishes a user's model from the solver in [this licensing explanation](https://lists.gnu.org/archive/html/help-glpk/2011-10/msg00000.html). Running `glpsol` as a separate program on your model differs from distributing an application linked with the GLPK library. Redistribution of GLPK, modifications or a combined work can trigger source-code obligations to recipients; private use alone does not require public release. See the [GNU GPL FAQ](https://www.gnu.org/licenses/gpl-faq.en.html#GPLRequireSourcePostedPublic). For a distributed product, ask your organization's licensing specialist to review the actual integration and distribution plan.

### Commercial solvers: community, trial and academic access

The main no-cost routes are compared below. Size limits apply to the model sent to the solver, including auxiliary variables and constraints introduced by reformulations. Consult the linked vendor terms for current limits and permitted uses.

| Solver | Community or trial access | Academic access |
|---|---|---|
| Gurobi | The bundled restricted license is for non-production use: up to 2,000 variables and 2,000 linear constraints, reduced to 200 variables when quadratic terms are present. See [limits](https://support.gurobi.com/hc/en-us/articles/360051597492-How-do-I-resolve-a-Model-too-large-for-size-limited-Gurobi-license-error). | Free licenses without model-size limits for eligible students, faculty and staff; named-user, WLS and institutional options. See [academic licenses](https://www.gurobi.com/academics). |
| CPLEX | Community Edition allows up to 1,000 variables and 1,000 constraints. | IBM's academic program provides eligible students and faculty access without functional or model-size limits. See [IBM's licensing comparison](https://www.ibm.com/products/ilog-cplex-optimization-studio/pricing). |
| MOSEK | A [30-day trial](https://www.mosek.com/products/trial/) has no model-size restrictions. | The free [personal academic license](https://www.mosek.com/products/academic-licenses/) has no size limit, lasts 365 days and is renewable; institutional floating licenses are also available. |
| Xpress | The bundled Community License allows **5,000 variables and constraints combined** for linear, quadratic and conic models. For general nonlinear models, FICO specifies a limit of **“200 variables and constraints”**. See [FICO's limits](https://github.com/fico-xpress/xpress-training/blob/main/python/md/python-full-course.md#chapter-2-installing-the-xpress-python-module) and the [combined counting rule](https://gams.com/53/docs/UG_License.html). | Consult [FICO's academic programs](https://community.fico.com/s/academic-programs) or your institution for the applicable access and terms. |

An academic license is permission for eligible educational or research work, not a general commercial entitlement. For example, [Gurobi](https://www.gurobi.com/academics) and [MOSEK](https://docs.mosek.com/license/license.pdf) restrict academic use to education and non-commercial research. An institutional email helps establish eligibility; it does not by itself authorize consulting, company deployment or every industry-funded project. Check the intended use with the vendor or your institution, along with renewal, machine/cloud access and concurrent-user rules. Keep personal license files and credentials private; a shared notebook should contain setup instructions, not somebody else's key.

## Installing and checking your setup

The next cell installs Pyomo and HiGHS on Colab. When running locally, install them first in the Python environment associated with your notebook using the command at the top of this page.

In [1]:
import sys

if "google.colab" in sys.modules:
    %pip install pyomo highspy

This example follows the book's `SolverFactory` pattern and checks an actual integer solution. It maximizes a nonnegative integer subject to twice its value being at most five.

In [2]:
import pyomo.environ as pyo

solver = "appsi_highs"
SOLVER = pyo.SolverFactory(solver)
assert SOLVER.available(), f"Solver {solver} is not available."

model = pyo.ConcreteModel()
model.x = pyo.Var(domain=pyo.NonNegativeIntegers)
model.capacity = pyo.Constraint(expr=2 * model.x <= 5)
model.profit = pyo.Objective(expr=model.x, sense=pyo.maximize)

results = SOLVER.solve(model)
pyo.assert_optimal_termination(results)
print(pyo.value(model.x))  # 2.0

2.0


The result is `2.0`: the best integer value is 2, whereas the continuous relaxation would allow 2.5.

The remaining installation commands and option snippets are reference instructions for other solvers and their notebooks. They are not needed to run this check; in particular, the MA57 option belongs to an Ipopt solver, not the HiGHS solver above.


For the additional solvers:

- **Ipopt and CBC:** the book often obtains executables through IDAES. Install with `python -m pip install pyomo idaes-pse`, then run `idaes get-extensions`. With the default installation location, `import idaes` configures the solver paths. The [IDAES guide](https://idaes-pse.readthedocs.io/en/stable/tutorials/getting_started/binaries.html) covers supported platforms and the included Bonmin and Couenne binaries. For preambles that download binaries with `--to ./bin`, use `os.environ["PATH"] += os.pathsep + os.path.abspath("bin")` after `import os`, replacing the Unix-only `":bin"` update so it also works on Windows.

  **Ipopt options.** Pyomo's shell-based solver interfaces set options through `SOLVER.options[...]`; option names and values depend on the solver. **Ipopt/MA57** means Ipopt using HSL's MA57 routine for its internal linear equations. If Ipopt reports insufficient memory with MA27, select MA57 on the Ipopt `SOLVER` before solving, provided your build includes it (see [Ipopt options](https://coin-or.github.io/Ipopt/OPTIONS.html#OPT_linear_solver)). The HSL-enabled IDAES binaries include the required license; availability varies by platform, and `"mumps"` is another choice where included.

  ```python
  SOLVER.options["linear_solver"] = "ma57"
  ```

  *HSL, a collection of Fortran codes for large-scale scientific computation. See [www.hsl.rl.ac.uk](https://www.hsl.rl.ac.uk/).*

- **MOSEK:** install with `python -m pip install pyomo mosek` and follow the [license setup instructions](https://docs.mosek.com/latest/install/installation.html). Eligible readers can request a [personal academic license](https://www.mosek.com/products/academic-licenses/) using their institutional email. Keep the license private. On each fresh Colab runtime, upload your `.lic` file and, after `import os`, set `os.environ["MOSEKLM_LICENSE_FILE"] = "/content/mosek.lic"` (adjust the path) before importing MOSEK, as explained in the [Colab guidance](https://docs.mosek.com/latest/faq/faq.html#how-to-use-mosek-in-a-google-colab-notebook).
- **SCIP:** for the Pyomo examples, install `python -m pip install "pyomo>=6.10.1" "pyscipopt>=6.2.1"` and use `pyo.SolverFactory("scip_direct")`. Supported binary packages include SCIP, as described in the [PySCIPOpt installation guide](https://pyscipopt.readthedocs.io/en/latest/install.html); this interface does not need a separate SCIP executable. §10.2 also uses PySCIPOpt directly. On Windows, install `pywin32` in the notebook environment (`%pip install pywin32`) and restart the kernel so Pyomo can use its Windows pipe support when capturing SCIP output. This prevents output-capture stalls that can occur even with `tee=False`.
- **Gurobi:** on Colab, run `%pip install pyomo gurobipy`. The bundled license handles small examples; larger models need a full license. **WLS (Web License Service)** supports hosted Colab, including [academic WLS access](https://www.gurobi.com/academics) for eligible readers. The [Web License Manager](https://license.gurobi.com/) is the portal where you create its API key and download the private `gurobi.lic` file; follow the [WLS setup guide](https://support.gurobi.com/hc/en-us/articles/13232844297489-How-do-I-set-up-a-Web-License-Service-WLS-license). WLS needs an internet connection to obtain license tokens.

  Upload `gurobi.lic` through Colab's **Files** pane on each fresh runtime. Then this optional example solves the integer `model` above with Gurobi; adjust the path if needed. Keep the file and its credentials out of shared notebook cells, outputs and repositories.

  ```python
  import os

  os.environ["GRB_LICENSE_FILE"] = "/content/gurobi.lic"

  with pyo.SolverFactory(
      "gurobi_direct", manage_env=True, options={"OutputFlag": 0}
  ) as SOLVER:
      results = SOLVER.solve(model)
      pyo.assert_optimal_termination(results)
  print(pyo.value(model.x))  # 2.0
  ```

  `gurobi_direct` uses the installed Python package. With [`manage_env=True`](https://pyomo.readthedocs.io/en/stable/api/pyomo.solvers.plugins.solvers.gurobi_direct.GurobiDirect.html), leaving the `with` block closes its models and environment. `OutputFlag` is set before startup to [suppress license details as well as solver logs](https://support.gurobi.com/hc/en-us/articles/360044784552-How-do-I-suppress-all-console-output-from-Gurobi). When finished, remove the uploaded file and choose **Runtime → Disconnect and delete runtime**; merely closing a browser tab can leave a WLS session active.

  Alternatively, use a [Colab local runtime](https://support.gurobi.com/hc/en-us/articles/4409582394769-Google-Colab-Installation-and-Licensing) on a suitably licensed machine. A laptop license does not automatically license hosted Colab. The table links to setup and licensing information for CPLEX and Xpress.
- **CVXPY:** install with `python -m pip install cvxpy`; its [installation guide](https://www.cvxpy.org/install/) explains additional solver options.

### Before you trust a result

Check solver termination before interpreting values. For MILO and global nonlinear optimization, report the feasible incumbent together with the bound or optimality gap when the solve stops early. For nonconvex problems, a successful local solve does not establish global optimality. Recording the solver version, settings and data helps others reproduce and interpret your results.

We thank [@leonlan](https://github.com/leonlan) for suggesting this overview in [issue #73](https://github.com/mobook/MO-book/issues/73).